#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [1]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangGraph와 LangChain – 한눈에 보는 차이점

| 구분 | LangChain | LangGraph |
|------|-----------|-----------|
| **주된 목적** | LLM(대형 언어 모델)을 활용한 **시퀀셜 파이프라인**(Chain), **에이전트** 및 **메모리**를 손쉽게 구성·운영 | **그래프 기반** 워크플로우 엔진으로 **상태(state)‑드리븐** 흐름, **조건·반복·분기** 등을 선언적으로 정의 |
| **핵심 추상화** | `Chain`, `Agent`, `Tool`, `Memory` 등 **선형·계층적** 구조 | `Node`, `Edge`, `State`, `Graph` 등 **노드‑엣지** 형태의 **유향 그래프** |
| **제어 흐름** | 기본적으로 **정적**(정해진 순서) 흐름. `if/else`, `while` 등은 파이썬 코드 안에서 직접 구현 | **동적**(런타임에 변하는) 흐름을 **그래프**와 **상태 전이**로 선언. 조건부 분기, 루프, 재시도 등을 그래프 자체에 명시 |
| **상태 관리** | 메모리 객체(`ConversationBufferMemory`, `RedisCache`, 등)로 **대화/컨텍스트**를 저장. 상태 전이는 별도 로직 필요 | **State 객체**가 그래프 전체에 자동으로 전파되고, 노드 간에 **읽기/쓰기**가 가능. 상태 변화가 그래프 실행에 직접 영향을 줌 |
| **디버깅·시각화** | 로그, 트레이싱, LangChain Hub에 예시 체인 제공. 시각화 도구는 별도 구현 필요 | `graphviz`, `mermaid` 등으로 **그래프 시각화**가 기본 제공. 실행 흐름을 실시간으로 확인 가능 |
| **재사용성** | 체인·에이전트·툴을 **모듈** 단위로 재사용. 하지만 복잡한 흐름을 재조합하려면 코드를 다시 짜야 함 | **노드**와 **서브그래프**를 조합해 복합 워크플로우를 손쉽게 재사용·조합 가능 |
